# NLP: воспроизводимый аудит сигналов переписки
Исправленная версия из последнего валидного коммита 3ef3b71. Вычисляет **новые** оценки, не перезаписывает старые `chat_features.csv` и `good_examples_of_verified_by_messages.csv`. Кандидаты обоих знаков — только для проверки, не ground truth. Отдельно учитываем факт продажи, который сам по себе не доказывает канал сделки. Не нужен для первого запуска model.ipynb.

In [1]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name in {"analysis", "feature", "models"}:
    repo_root = repo_root.parent

assert (repo_root / "requirements.txt").is_file() and (repo_root / "data/listings.csv").is_file(), (
    "Запустите ноутбук из корня репозитория или из папки, в которой он лежит"
)
os.chdir(repo_root)

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from IPython.display import display
import torch

messages = pd.read_csv("data/data_wthout_anomailes/messages_wthout_anomailes.csv")
messages["sent_at"] = pd.to_datetime(messages["sent_at"], errors="coerce", format="mixed")
messages = messages.dropna(subset=["listing_id", "chat_id", "sent_at", "text"]).copy()
messages["text"] = messages["text"].astype(str).str.strip()
messages = messages[messages["text"].ne("")]
assert messages.groupby("chat_id")["listing_id"].nunique().max() == 1
chat_ids = messages[["chat_id", "listing_id"]].drop_duplicates()
messages = messages.sort_values("sent_at", ascending=False, kind="stable").groupby("chat_id", sort=False).head(5).copy()
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda" if torch.cuda.is_available() else "cpu")
print(model.device)


c:\Users\sbitw\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5366.63it/s]


cpu


In [4]:

sale_signals = {

    # ==================================================
    # POSITIVE: ПРИЗНАКИ ПОТЕНЦИАЛЬНОЙ И СОСТОЯВШЕЙСЯ СДЕЛКИ
    # ==================================================

    "purchase_intent": [
        "Я хочу купить этот товар",
        "Готов приобрести ваш товар",
        "Меня интересует покупка",
        "Я заинтересован в приобретении",
        "Хочу забрать этот товар",
        "Готов забрать прямо сегодня",
        "Могу купить прямо сейчас",
        "Я согласен на покупку",
        "Хочу приобрести именно этот товар",
        "Можно оформить покупку?",
        "Я беру этот товар",
        "Давайте оформим сделку",
        "Готов забрать без лишних вопросов",
        "Мне подходит ваше предложение",
        "Я готов приобрести его",
        "Хочу купить, если товар ещё доступен",
        "Договорились, я покупаю",
        "Могу приехать и забрать",
        "Готов забрать за указанную цену",
        "Я точно заинтересован в покупке",
        "Хочу забрать его себе",
        "Покупка меня интересует",
        "Давайте договоримся о покупке",
        "Готов приобрести без торга",
        "Я согласен приобрести товар"
    ],

    "seller_confirmation": [
        "Да, товар ещё продаётся",
        "Да, объявление актуально",
        "Товар в наличии",
        "Да, можете забрать",
        "Товар пока не продан",
        "Ещё не продал, можете приехать",
        "Да, всё доступно",
        "Товар свободен",
        "Можете забрать в любое удобное время",
        "Да, продаю",
        "Объявление актуально, пишите",
        "Товар пока у меня",
        "Да, можно договориться о встрече",
        "Могу продать вам",
        "Товар ещё доступен для покупки",
        "Да, готов продать",
        "Можете приезжать за ним",
        "Пока никто не забрал",
        "Да, товар не забронирован",
        "Готов встретиться для продажи"
    ],

    "price_negotiation": [
        "Можно немного уступить в цене?",
        "Сделаете небольшую скидку?",
        "Готов купить за меньшую сумму",
        "Давайте договоримся по цене",
        "Какая минимальная цена?",
        "Могу предложить другую сумму",
        "Согласны продать дешевле?",
        "Если заберу сегодня, сделаете скидку?",
        "Давайте сойдёмся на этой цене",
        "Предлагаю встретиться посередине",
        "Готов заплатить указанную сумму",
        "Цена меня устраивает",
        "Согласен на вашу цену",
        "Договорились по стоимости",
        "Хорошо, беру за эту сумму",
        "Сколько будет стоить с доставкой?",
        "Могу перевести задаток",
        "За эту цену готов забрать",
        "Давайте зафиксируем стоимость",
        "Хорошо, договорились о цене"
    ],

    "reservation": [
        "Отложите товар для меня",
        "Можно забронировать товар?",
        "Забронируйте, пожалуйста, до вечера",
        "Я точно приеду, придержите товар",
        "Можете пока никому не продавать?",
        "Готов внести предоплату за бронь",
        "Оставьте товар за мной",
        "Давайте договоримся о резерве",
        "Можно забрать завтра, придержите?",
        "Я приеду за забронированным товаром",
        "Снимите объявление, я его забираю",
        "Давайте зафиксируем договорённость",
        "Товар остаётся за мной?",
        "Подержите его до выходных",
        "Я подтверждаю бронь"
    ],

    "meeting": [
        "Договорились встретиться сегодня в шесть",
        "Я на месте, подхожу за товаром",
        "Встретимся завтра у метро, время подтверждаю",
        "Когда можно приехать за товаром?",
        "Где можно встретиться?",
        "Давайте договоримся о встрече",
        "Могу подъехать сегодня вечером",
        "Во сколько вам удобно встретиться?",
        "Напишите адрес, куда подъехать",
        "Я буду у вас через час",
        "Давайте встретимся возле метро",
        "Могу приехать завтра утром",
        "Когда вам удобно передать товар?",
        "Давайте согласуем время встречи",
        "Я уже выезжаю к вам",
        "Буду на месте через 15 минут",
        "Могу забрать товар лично",
        "Куда подъехать для покупки?",
        "Давайте встретимся сегодня",
        "Я подъехал, где вас найти?",
        "Встречаемся в оговорённом месте",
        "Подтвердите время встречи",
        "Хорошо, тогда до встречи"
    ],


    "payment": [
        "Куда перевести деньги?",
        "Как можно оплатить товар?",
        "Я готов оплатить покупку",
        "Деньги уже перевёл",
        "Оплата прошла успешно",
        "Перевёл вам указанную сумму",
        "Оплатил заказ",
        "Могу оплатить наличными",
        "Оплачу при получении",
        "Давайте согласуем способ оплаты",
        "Сколько нужно перевести?",
        "Оплата произведена полностью",
        "Вы получили перевод?",
        "Деньги отправлены на вашу карту",
        "Готов рассчитаться при встрече",
        "Я оплатил товар",
        "Получили оплату?",
        "Всё оплачено, спасибо",
        "Давайте завершим оплату",
        "Перевод отправлен, проверьте"
    ],

    "deal_confirmation": [
        "Передал товар покупателю, деньги получил",
        "Встретились и завершили сделку",
        "Спасибо, товар передан и оплачен",
        "Сделка состоялась, спасибо",
        "Спасибо за покупку",
        "Спасибо за продажу",
        "Договорённость выполнена",
        "Товар продан, спасибо",
        "Всё, сделка завершена",
        "Спасибо, всё получил",
        "Покупка успешно состоялась",
        "Деньги получил, спасибо",
        "Спасибо за оперативную сделку",
        "Договорились, товар теперь мой",
        "Спасибо, сделка завершена",
        "Покупатель забрал товар",
        "Продажа прошла успешно",
        "Спасибо, рад покупке"
    ],

    "item_received": [
        "Забрал товар, спасибо",
        "Посылку получил, всё в порядке",
        "Получила заказ, спасибо продавцу",
        "Товар получил, спасибо",
        "Всё забрал, благодарю",
        "Получил посылку",
        "Забрал заказ из пункта выдачи",
        "Товар уже у меня",
        "Всё получил в целости",
        "Посылка пришла, спасибо",
        "Получил покупку, всё отлично",
        "Забрал товар, всё хорошо",
        "Спасибо, товар доставлен",
        "Получение прошло успешно",
        "Забрал сегодня, спасибо",
        "Товар соответствует описанию",
        "Покупкой доволен",
        "Всё на месте, спасибо"
    ],


    # ==================================================
    # NEGATIVE: ПРИЗНАКИ НЕСОСТОЯВШЕЙСЯ СДЕЛКИ
    # ==================================================

    "buyer_refusal": [
        "Я передумал покупать",
        "Больше не интересует",
        "Покупать не буду",
        "Отказываюсь от покупки",
        "Мне больше не нужен этот товар",
        "Я решил не приобретать",
        "Не буду забирать товар",
        "Пока откажусь от предложения",
        "Нашёл другой вариант",
        "Покупка отменяется с моей стороны",
        "Извините, не получится купить",
        "Я больше не заинтересован",
        "Передумал, спасибо",
        "Не готов совершать покупку",
        "Спасибо, но я откажусь",
        "Уже не актуально для меня",
        "Не буду приезжать",
        "Решил пока ничего не покупать",
        "Вынужден отказаться от покупки",
        "Пожалуй, не буду брать"
    ],

    
    "seller_sold": [
        "Я уже продал этот товар",
        "Товар уже продан",
        "Продал его другому покупателю",
        "Покупатель уже забрал товар",
        "Товар больше не продаётся",
        "Уже продано, извините",
        "Продажа состоялась",
        "За товаром уже приезжали",
        "Товар забрали сегодня",
        "Уже нашёл покупателя",
        "Продал, спасибо за интерес",
        "Объявление больше не актуально, товар продан",
        "Вещь уже у нового владельца",
        "Продал через объявление",
        "Покупатель забрал его вчера",
        "Спасибо, товар уже нашёл своего покупателя"
    ],


    "seller_refusal": [
        "Я не готов продавать",
        "Не хочу продавать за такую цену",
        "Отказываюсь от сделки",
        "Не буду продавать вам товар",
        "Продажа отменяется",
        "Я решил оставить товар себе",
        "Не готов отдавать товар",
        "Не могу продать по этой цене",
        "Извините, продавать передумал",
        "Не получится продать вам",
        "Я не согласен на предложенные условия",
        "Товар продавать больше не планирую",
        "Пока не готов к продаже",
        "Решил не расставаться с вещью",
        "Не буду заключать сделку"
    ],

    "price_disagreement": [
        "Ваша цена меня не устраивает",
        "Не договорились по стоимости",
        "Слишком дорого для меня",
        "За такую сумму покупать не буду",
        "Продавец не согласился на скидку",
        "Мы не сошлись в цене",
        "Не готов платить столько",
        "Цена слишком высокая",
        "Не получается договориться о стоимости",
        "Не согласен с предложенной ценой",
        "Торг не состоялся",
        "Не договорились о скидке",
        "Предложенная сумма слишком маленькая",
        "Продавец не хочет уступать",
        "На таких условиях сделка невозможна"
    ],

    "meeting_failure": [
        "Приехал на встречу, но продавец не пришёл",
        "Покупатель не приехал за товаром",
        "Встреча не состоялась",
        "Не смог приехать за товаром",
        "Продавец не пришёл на встречу",
        "Покупатель не приехал",
        "Договорённость о встрече сорвалась",
        "Не удалось встретиться",
        "Вы не пришли в назначенное время",
        "Я не смогу подъехать",
        "Не получилось забрать товар",
        "Встречу пришлось отменить",
        "Не дождался продавца",
        "Не получилось согласовать время",
        "Встреча переносится на неопределённый срок",
        "Я не приехал за товаром",
        "Передача товара не состоялась"
    ],

    "cancellation": [
        "Встречу отменяем, сделка не состоится",
        "Давайте отменим договорённость",
        "Я отменяю заказ",
        "Покупка отменена",
        "Договорённость больше не актуальна",
        "Не будем продолжать сделку",
        "Давайте прекратим обсуждение",
        "Отмените, пожалуйста, бронь",
        "Больше не нужно резервировать товар",
        "Сделку пришлось отменить",
        "Я отменил оформление",
        "Не будем ничего оформлять",
        "Заказ отменён",
        "Решил отменить покупку",
        "Договорённость аннулируется",
        "Не получится завершить сделку"
    ],

    "item_unavailable": [
        "Товара больше нет в наличии",
        "Товар уже забрали",
        "Объявление больше не актуально",
        "Этот товар больше не продаётся",
        "Уже нет возможности купить",
        "Товар продан другому человеку",
        "Вещь уже недоступна",
        "К сожалению, товар закончился",
        "Товар больше не у меня",
        "Его уже забрали",
        "Больше не могу предложить этот товар",
        "Товар снят с продажи",
        "Продажа этого товара прекращена",
        "Товар уже передан другому покупателю",
        "Больше не принимаю предложения по этому товару"
    ],

    "external_sale": [
        "Продал товар знакомому",
        "Продал не через Авито",
        "Продал на другой площадке",
        "Продал через социальные сети",
        "Товар забрал мой знакомый",
        "Продал родственнику",
        "Нашёл покупателя самостоятельно",
        "Продал через другой сайт",
        "Договорился с покупателем вне площадки",
        "Продал человеку из своего окружения",
        "Товар купил мой коллега",
        "Продал через знакомых",
        "Покупатель нашёлся не на Авито",
        "Продал через другой сервис",
        "Уже продал вне площадки"
    ],

    "seller_changed_mind": [
        "Передумал продавать",
        "Решил оставить товар себе",
        "Больше не хочу расставаться с вещью",
        "Объявление снимаю, продавать не буду",
        "Пока оставлю товар у себя",
        "Решил не продавать",
        "Товар мне ещё нужен",
        "Продажа больше не актуальна",
        "Передумал выставлять товар на продажу",
        "Пока не планирую продавать",
        "Оставлю вещь для личного использования",
        "Решил отложить продажу",
        "Больше не рассматриваю продажу",
        "Не хочу сейчас продавать эту вещь",
        "Сниму объявление, товар оставлю себе"
    ]
}



In [5]:
# Сходство с фразами каждой категории, без суммирования десятков
# заведомо ненулевых косинусных сходств.
texts, inverse = np.unique(messages["text"].to_numpy(dtype=str), return_inverse=True)
emb = model.encode(texts.tolist(), normalize_embeddings=True, batch_size=128, show_progress_bar=True)
for category, phrases in sale_signals.items():
    anchors = model.encode(phrases, normalize_embeddings=True, show_progress_bar=False)
    messages[category] = (emb @ anchors.T).max(axis=1)[inverse]

positive = ["deal_confirmation", "item_received"]
negative = ["buyer_refusal", "seller_refusal", "meeting_failure", "cancellation", "external_sale", "seller_changed_mind"]
# Намерение, цена, договорённость и generic seller_sold не равны завершению сделки.
other = [c for c in sale_signals if c not in positive + negative]
p = messages[positive].max(axis=1)
n = messages[negative].max(axis=1)
o = messages[other].max(axis=1)
# Эвристические пороги, не подогнаны под verified; подлежат независимой проверке.
MIN_SIMILARITY = 0.75
MIN_MARGIN = 0.06
messages["positive_evidence"] = p.where((p >= MIN_SIMILARITY) & (p - np.maximum(n, o) >= MIN_MARGIN), 0)
messages["negative_evidence"] = n.where((n >= MIN_SIMILARITY) & (n - np.maximum(p, o) >= MIN_MARGIN), 0)
chat_scores = messages.groupby(["listing_id", "chat_id"], as_index=False).agg(
    scored_message_count=("text", "size"),
    strongest_positive_signal=("positive_evidence", "max"),
    strongest_negative_signal=("negative_evidence", "max"),
)
chat_scores["deal_signal_score"] = chat_scores["strongest_positive_signal"] - chat_scores["strongest_negative_signal"]
chat_scores["deal_signal_strength"] = chat_scores[["strongest_positive_signal", "strongest_negative_signal"]].max(axis=1)
assert chat_scores["chat_id"].is_unique
chat_scores.to_csv("feature/chat_features_audit.csv", index=False)
# Объявление может иметь несколько чатов: негатив в одном не означает отсутствия сделки.
# Сохраняем кандидатов на ручную проверку на уровне чата, а не готовые labels.
review = chat_scores.loc[(chat_scores["strongest_positive_signal"] > 0) ^ (chat_scores["strongest_negative_signal"] > 0)].copy()
review["candidate_chat_outcome"] = (review["strongest_positive_signal"] > 0).astype(int)
review = review.sort_values(["deal_signal_strength", "listing_id", "chat_id"], ascending=[False, True, True], kind="stable")
review.to_csv("feature/chat_review_candidates_audit.csv", index=False)
print("Оценённых чатов:", len(chat_scores))
print("Кандидаты на проверку:")
display(review["candidate_chat_outcome"].value_counts())


Batches: 100%|██████████| 81/81 [00:12<00:00,  6.41it/s]


Оценённых чатов: 88576
Кандидаты на проверку:


candidate_chat_outcome
0    2044
1     600
Name: count, dtype: int64